# 06 Baseline Comparisons

Evaluate simpler anomaly detectors on the same frozen split as the CNN autoencoder. Thresholds are selected on validation data and then frozen for the test split.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from PIL import Image
from scipy.signal import welch
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM


In [2]:
REPO_ROOT = Path('..').resolve()
DATASET_CONFIGS = {
    'turning': {
        'manifest_path': REPO_ROOT / 'reports' / 'manifests' / 'turning_split_seed42.csv',
        'nominal_label': 'no_chatter',
        'positive_label': 'chatter',
        'sample_rate_hz': 50000,
    },
    'cnc_machining': {
        'manifest_path': REPO_ROOT / 'reports' / 'manifests' / 'cnc_machining_split_seed42.csv',
        'nominal_label': 'nominal',
        'positive_label': 'anomaly',
        'sample_rate_hz': 2000,
    },
}
DATASETS_TO_EVALUATE = tuple(DATASET_CONFIGS)
BASELINE_DIR = REPO_ROOT / 'reports' / 'baselines'
THRESHOLD_DIR = REPO_ROOT / 'reports' / 'thresholds'
TABLE_DIR = REPO_ROOT / 'reports' / 'tables'

IMAGE_SIZE = (150, 100)
MAX_FREQ = 5000
PCA_COMPONENTS = 32
SEED = 42

BASELINE_DIR.mkdir(parents=True, exist_ok=True)
THRESHOLD_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
manifest_frames = []
for dataset_name in DATASETS_TO_EVALUATE:
    config = DATASET_CONFIGS[dataset_name]
    dataset_manifest = pd.read_csv(config['manifest_path'])
    dataset_manifest = dataset_manifest[dataset_manifest['source_dataset'] == dataset_name].copy()
    dataset_manifest['target'] = (dataset_manifest['label'] == config['positive_label']).astype(int)
    manifest_frames.append(dataset_manifest)

manifest = pd.concat(manifest_frames, ignore_index=True)
manifest.groupby(['source_dataset', 'split', 'label']).size().rename('n').reset_index()


,source_dataset,split,label,n
0,cnc_machining,test,anomaly,481
1,cnc_machining,test,nominal,5144
2,cnc_machining,train,nominal,23698
3,cnc_machining,validation,anomaly,612
4,cnc_machining,validation,nominal,4975
5,turning,test,chatter,27
6,turning,test,no_chatter,48
7,turning,train,no_chatter,472
8,turning,validation,chatter,34
9,turning,validation,no_chatter,70


In [3]:
def spectral_features(signal: np.ndarray, fs: int, max_freq: int = MAX_FREQ) -> dict:
    freqs, psd = welch(signal, fs=fs, nperseg=min(2048, len(signal)))
    mask = freqs <= max_freq
    freqs = freqs[mask]
    psd = psd[mask]
    total = float(np.sum(psd) + 1e-12)
    centroid = float(np.sum(freqs * psd) / total)
    bandwidth = float(np.sqrt(np.sum(((freqs - centroid) ** 2) * psd) / total))
    dominant_frequency = float(freqs[int(np.argmax(psd))]) if len(freqs) else np.nan
    bands = [(0, 1000), (1000, 2000), (2000, 3000), (3000, 4000), (4000, 5000)]
    band_values = {f'band_{lo}_{hi}': float(np.sum(psd[(freqs >= lo) & (freqs < hi)])) for lo, hi in bands}
    return {'spectral_centroid': centroid, 'spectral_bandwidth': bandwidth, 'dominant_frequency': dominant_frequency, **band_values}

def window_features(npz_path: Path, fallback_fs: int) -> dict:
    with np.load(npz_path) as data:
        fs = int(float(data['fs'])) if 'fs' in data.files else fallback_fs
        features = {}
        for axis in ['X', 'Y', 'Z']:
            values = data[axis].astype(float)
            rms = float(np.sqrt(np.mean(values ** 2)))
            peak = float(np.max(np.abs(values)))
            features[f'{axis}_rms'] = rms
            features[f'{axis}_peak_abs'] = peak
            features[f'{axis}_crest_factor'] = peak / max(rms, 1e-12)
            features.update({f'{axis}_{k}': v for k, v in spectral_features(values, fs=fs).items()})
    return features

def image_vector(image_path: Path) -> np.ndarray:
    image = Image.open(image_path).convert('RGB').resize(IMAGE_SIZE)
    return np.asarray(image, dtype='float32').reshape(-1) / 255.0


In [4]:
feature_rows = []
image_rows = []
for _, row in manifest.iterrows():
    config = DATASET_CONFIGS[row['source_dataset']]
    npz_path = REPO_ROOT / row['npz_path']
    feature_rows.append({'source_dataset': row['source_dataset'], 'sample_id': row['sample_id'], **window_features(npz_path, config['sample_rate_hz'])})
    if isinstance(row.get('image_path'), str) and row['image_path']:
        image_rows.append({'source_dataset': row['source_dataset'], 'sample_id': row['sample_id'], 'image_vector': image_vector(REPO_ROOT / row['image_path'])})

feature_table = manifest.merge(pd.DataFrame(feature_rows), on=['source_dataset', 'sample_id'], how='left')
if image_rows:
    image_table = manifest.merge(pd.DataFrame(image_rows), on=['source_dataset', 'sample_id'], how='inner')
else:
    image_table = manifest.iloc[0:0].copy()
print('feature rows:', len(feature_table))
print('image rows:', len(image_table))


feature rows: 37367
image rows: 37367


In [5]:
def select_best_f1_threshold(y_true: np.ndarray, scores: np.ndarray) -> dict:
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best_idx = int(np.nanargmax(f1))
    return {'threshold': float(thresholds[best_idx]), 'validation_f1': float(f1[best_idx])}

def evaluate_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'pr_auc': float(average_precision_score(y_true, scores)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


In [6]:
baseline_metrics = []
baseline_score_frames = []
baseline_thresholds = {dataset_name: {} for dataset_name in DATASETS_TO_EVALUATE}

for dataset_name in DATASETS_TO_EVALUATE:
    config = DATASET_CONFIGS[dataset_name]
    dataset_features = feature_table[feature_table['source_dataset'] == dataset_name].copy()
    feature_columns = [c for c in dataset_features.columns if c.startswith(('X_', 'Y_', 'Z_'))]
    train_nominal = dataset_features[(dataset_features['split'] == 'train') & (dataset_features['label'] == config['nominal_label'])]
    validation = dataset_features[dataset_features['split'] == 'validation']
    test = dataset_features[dataset_features['split'] == 'test']
    if train_nominal.empty or validation.empty or test.empty:
        print(f'Skipping {dataset_name}: missing train/validation/test rows.')
        continue

    scaler = StandardScaler().fit(train_nominal[feature_columns])
    X_train = scaler.transform(train_nominal[feature_columns])
    X_val = scaler.transform(validation[feature_columns])
    X_test = scaler.transform(test[feature_columns])
    y_val = validation['target'].to_numpy()
    y_test = test['target'].to_numpy()

    ocsvm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05).fit(X_train)
    iforest = IsolationForest(random_state=SEED, contamination='auto').fit(X_train)
    models = {
        'one_class_svm_features': lambda x, model=ocsvm: -model.decision_function(x),
        'isolation_forest_features': lambda x, model=iforest: -model.decision_function(x),
    }

    for method, score_fn in models.items():
        val_scores = score_fn(X_val)
        test_scores = score_fn(X_test)
        selected = select_best_f1_threshold(y_val, val_scores)
        baseline_thresholds[dataset_name][method] = selected
        metrics = evaluate_at_threshold(y_test, test_scores, selected['threshold'])
        baseline_metrics.append({'dataset': dataset_name, 'method': method, 'score': 'anomaly_score', 'threshold': selected['threshold'], **metrics})

        for split_name, rows, targets, scores in [
            ('validation', validation, y_val, val_scores),
            ('test', test, y_test, test_scores),
        ]:
            baseline_score_frames.append(pd.DataFrame({
                'dataset': dataset_name,
                'method': method,
                'sample_id': rows['sample_id'].to_numpy(),
                'split': split_name,
                'label': rows['label'].to_numpy(),
                'target': targets,
                'score_value': scores,
            }))


In [7]:
# Feature-space baselines are computed in the previous cell for each dataset.


In [8]:
for dataset_name in DATASETS_TO_EVALUATE:
    config = DATASET_CONFIGS[dataset_name]
    dataset_images = image_table[image_table['source_dataset'] == dataset_name].copy()
    image_train = dataset_images[(dataset_images['split'] == 'train') & (dataset_images['label'] == config['nominal_label'])]
    image_val = dataset_images[dataset_images['split'] == 'validation']
    image_test = dataset_images[dataset_images['split'] == 'test']
    if image_train.empty or image_val.empty or image_test.empty:
        print(f'Skipping PCA image baseline for {dataset_name}: missing generated images.')
        continue

    X_img_train = np.stack(image_train['image_vector'].to_numpy())
    X_img_val = np.stack(image_val['image_vector'].to_numpy())
    X_img_test = np.stack(image_test['image_vector'].to_numpy())
    y_img_val = image_val['target'].to_numpy()
    y_img_test = image_test['target'].to_numpy()

    pca = PCA(n_components=min(PCA_COMPONENTS, X_img_train.shape[0], X_img_train.shape[1]), random_state=SEED).fit(X_img_train)
    val_recon = pca.inverse_transform(pca.transform(X_img_val))
    test_recon = pca.inverse_transform(pca.transform(X_img_test))
    val_scores = np.mean((X_img_val - val_recon) ** 2, axis=1)
    test_scores = np.mean((X_img_test - test_recon) ** 2, axis=1)
    method = 'pca_image_reconstruction'
    selected = select_best_f1_threshold(y_img_val, val_scores)
    baseline_thresholds[dataset_name][method] = selected
    metrics = evaluate_at_threshold(y_img_test, test_scores, selected['threshold'])
    baseline_metrics.append({'dataset': dataset_name, 'method': method, 'score': 'reconstruction_mse', 'threshold': selected['threshold'], **metrics})

    for split_name, rows, targets, scores in [
        ('validation', image_val, y_img_val, val_scores),
        ('test', image_test, y_img_test, test_scores),
    ]:
        baseline_score_frames.append(pd.DataFrame({
            'dataset': dataset_name,
            'method': method,
            'sample_id': rows['sample_id'].to_numpy(),
            'split': split_name,
            'label': rows['label'].to_numpy(),
            'target': targets,
            'score_value': scores,
        }))


In [9]:
baseline_metrics = pd.DataFrame(baseline_metrics)
baseline_scores = pd.concat(baseline_score_frames, ignore_index=True) if baseline_score_frames else pd.DataFrame()

combined_metrics_path = TABLE_DIR / 'metrics_baselines_all_datasets.csv'
combined_scores_path = BASELINE_DIR / 'baseline_scores_all_datasets.csv'
baseline_metrics.to_csv(combined_metrics_path, index=False)
baseline_scores.to_csv(combined_scores_path, index=False)
print(f'Wrote {combined_metrics_path}')
print(f'Wrote {combined_scores_path}')

for dataset_name in DATASETS_TO_EVALUATE:
    dataset_metrics = baseline_metrics[baseline_metrics['dataset'] == dataset_name]
    dataset_scores = baseline_scores[baseline_scores['dataset'] == dataset_name] if not baseline_scores.empty else pd.DataFrame()
    if dataset_metrics.empty:
        continue
    metrics_path = TABLE_DIR / f'metrics_{dataset_name}_baselines.csv'
    scores_path = BASELINE_DIR / f'baseline_scores_{dataset_name}.csv'
    threshold_path = THRESHOLD_DIR / f'baseline_thresholds_{dataset_name}.json'
    dataset_metrics.to_csv(metrics_path, index=False)
    dataset_scores.to_csv(scores_path, index=False)
    with threshold_path.open('w') as f:
        json.dump(baseline_thresholds[dataset_name], f, indent=2)
    print(f'Wrote {metrics_path}')
    print(f'Wrote {scores_path}')
    print(f'Wrote {threshold_path}')

baseline_metrics


Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/tables/metrics_baselines_all_datasets.csv
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/baselines/baseline_scores_all_datasets.csv
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/tables/metrics_turning_baselines.csv
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/baselines/baseline_scores_turning.csv
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/thresholds/baseline_thresholds_turning.json
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/tables/metrics_cnc_machining_baselines.csv
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/baselines/baseline_scores_cnc_machining.csv
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/thresholds/baseline_thresholds_cnc_machining.json


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,fn,tp
0,turning,one_class_svm_features,anomaly_score,0.253464,0.959115,0.905660,0.923077,0.888889,46,2,3,24
1,turning,isolation_forest_features,anomaly_score,-0.039756,0.990143,0.928571,0.896552,0.962963,45,3,1,26
2,cnc_machining,one_class_svm_features,anomaly_score,-32.893099,0.139689,0.250213,0.143043,0.997730,21,5266,2,879
3,cnc_machining,isolation_forest_features,anomaly_score,-0.131895,0.149312,0.249124,0.142953,0.968218,173,5114,28,853
4,turning,pca_image_reconstruction,reconstruction_mse,0.001279,0.961881,0.947368,0.900000,1.000000,45,3,0,27
5,cnc_machining,pca_image_reconstruction,reconstruction_mse,0.000015,0.134950,0.256681,0.147411,0.992054,232,5055,7,874
